In [9]:
from model.unet import UNet
import torch
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from data.load_data import BreastUltrasoundDataset
import torch.nn as nn

In [10]:
dataset = BreastUltrasoundDataset(image_size=256)

In [11]:
# -------------------
# DEVICE
# -------------------
DEVICE = torch.device("cpu")

# -------------------
# LOAD MODEL
# -------------------
from model.unet import UNet

from model.unet import UNet

model = UNet(n_class=1)
model.load_state_dict(torch.load("model(8).bin", map_location=DEVICE))
model.to(DEVICE)
model.eval()

print("Model loaded successfully")



Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Model loaded successfully


In [16]:
import time
import torch
import psutil
import numpy as np

model.eval()

img, _ = dataset[0]

# Add batch dimension
img = img.unsqueeze(0).to("cpu")

# -----------------------------------
# Warmup
# -----------------------------------
with torch.no_grad():
    for _ in range(10):
        _ = model(img)

# -----------------------------------
# Benchmark
# -----------------------------------
repetitions = 100

times = []
cpu_freqs = []

with torch.no_grad():

    for _ in range(repetitions):

        # Current CPU frequency
        freq = psutil.cpu_freq().current
        cpu_freqs.append(freq)

        start = time.perf_counter()

        _ = model(img)

        end = time.perf_counter()

        times.append((end - start) * 1000)  # ms

# -----------------------------------
# Statistics
# -----------------------------------
times = np.array(times)
cpu_freqs = np.array(cpu_freqs)

avg_time = times.mean()
std_time = times.std()

avg_freq = cpu_freqs.mean()
max_freq = cpu_freqs.max()
min_freq = cpu_freqs.min()

fps = 1000 / avg_time

# -----------------------------------
# Results
# -----------------------------------
print("=" * 50)
print("CPU Inference Benchmark")
print("=" * 50)

print(f"Average inference time : {avg_time:.3f} ms")
print(f"Std inference time     : {std_time:.3f} ms")

print(f"FPS                    : {fps:.2f}")

print("-" * 50)

print(f"Average CPU clock      : {avg_freq:.0f} MHz")

print("=" * 50)

CPU Inference Benchmark
Average inference time : 40.285 ms
Std inference time     : 2.591 ms
FPS                    : 24.82
--------------------------------------------------
Average CPU clock      : 2300 MHz
